# Dataset Playground

This notebook loads the different items in the dataset for experimentation.

In [13]:
import pickle
import os
import json
import torch

DATASET_ROOT = '/data/nas-gpu/wang/atong/Datasets/MoonshotDatasetv3'

index = pickle.load(open(os.path.join(DATASET_ROOT, 'index.pkl'), 'rb'))
metadata = json.load(open(os.path.join(DATASET_ROOT, 'metadata.json'), 'r'))
counts = pickle.load(open(os.path.join(DATASET_ROOT, 'count_hashes_under_radius_6.pkl'), 'rb'))
retrieval = pickle.load(open(os.path.join(DATASET_ROOT, 'retrieval.pkl'), 'rb'))
bitinfo_to_idx = pickle.load(open(os.path.join(DATASET_ROOT, 'RankingEntropy', 'bitinfo_to_idx.pkl'), 'rb'))
rankingset = torch.load(os.path.join(DATASET_ROOT, 'RankingEntropy', 'rankingset.pt'))

In [14]:
import sys
import os
sys.path.append('/data/nas-gpu/wang/atong/SMART-Moonshot')
from src.modules.data.fp_loader import EntropyFPLoader
from src.modules.core.const import DATASET_ROOT
fp_loader = EntropyFPLoader(retrieval_path=os.path.join(DATASET_ROOT, 'retrieval.pkl'))
fp_loader.setup(out_dim=16384, max_radius=6)


from src.modules.marina import MARINA
from src.modules.marina.args import MARINAArgs
args = MARINAArgs()

model = MARINA(args, fp_loader)
model.setup_ranker()

2026-03-10 17:24:49,237 - modules.data.fp_loader - INFO - Setting up EntropyFPLoader...
2026-03-10 17:24:57,618 - modules.data.fp_loader - INFO - Done! Selected 16384 features in 8.38s.
2026-03-10 17:24:57,953 - modules.marina.model - INFO - [MARINA] Started Initializing
2026-03-10 17:24:58,494 - modules.marina.model - INFO - [MARINA] Initialized
2026-03-10 17:24:58,961 - modules.core.ranker - INFO - [RankingSet] Initialized with 518901 sample(s); metric=cosine


In [16]:
import rdkit.Chem as Chem

def canonicalize_smiles(smiles: str, keep_stereo: bool = False):
    if smiles is None or smiles == '':
        raise ValueError(f"Invalid empty SMILES")
    if '.' in smiles:
        smiles = max(smiles.split('.'), key=len)
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")
    return Chem.MolToSmiles(mol, isomericSmiles=keep_stereo, canonical=True)

In [18]:
from tqdm import tqdm
smiles = canonicalize_smiles('[H][C@]12[C@@H](C)[C@H](O)C3=CC(=O)C=C(N4C(=O)C(C)=C(O)[C@]4(CC)C=C1C)[C@]23O')
for idx, d in tqdm(retrieval.items()):
    if d['smiles'] == smiles:
        print(idx)
        break

 96%|█████████▋| 499590/518901 [00:00<00:00, 6627318.96it/s]

499590


In [19]:
rankingset[499590]

tensor(indices=tensor([[    5,     6,     7,    11,    13,    16,    19,    20,
                           21,    29,    34,    37,    50,    63,   133,   252,
                          275,   333,   434,   526,   560,   744,   762,   858,
                         1512,  1620,  2423,  3539, 12370, 14275]]),
       values=tensor([0.1826, 0.1826, 0.1826, 0.1826, 0.1826, 0.1826, 0.1826,
                      0.1826, 0.1826, 0.1826, 0.1826, 0.1826, 0.1826, 0.1826,
                      0.1826, 0.1826, 0.1826, 0.1826, 0.1826, 0.1826, 0.1826,
                      0.1826, 0.1826, 0.1826, 0.1826, 0.1826, 0.1826, 0.1826,
                      0.1826, 0.1826]),
       size=(16384,), nnz=30, layout=torch.sparse_coo)

In [20]:
import torch
fp = fp_loader.build_mfp_for_smiles('[H][C@]12[C@@H](C)[C@H](O)C3=CC(=O)C=C(N4C(=O)C(C)=C(O)[C@]4(CC)C=C1C)[C@]23O')
torch.nonzero(fp > 0).squeeze()

tensor([    5,     6,     7,    11,    13,    16,    19,    20,    21,    29,
           34,    37,    50,    63,   133,   252,   275,   333,   434,   526,
          560,   744,   762,   858,  1512,  1620,  2423,  3539, 12370, 14275])

In [21]:
model.ranker.retrieve_idx(fp, 10)

tensor([499590, 499594,  64742, 499593, 499596, 499592, 499595,  81995, 112016,
        338809])

In [22]:
retrieval[499590]

{'smiles': 'CCC12C=C(C)C3C(C)C(O)C4=CC(=O)C=C(N1C(=O)C(C)=C2O)C43O'}

In [24]:
import pickle
benchmark_data = pickle.load(open(os.path.join('/data/nas-gpu/wang/atong/Datasets/Benchmark', 'benchmark.pkl'), 'rb'))

In [26]:
benchmark_data[0]

{'input': {'h_nmr': tensor([[7.0500],
          [6.8600],
          [3.6600],
          [2.7900],
          [2.6800],
          [2.6800],
          [2.4700],
          [2.3600],
          [2.1200],
          [2.0000],
          [1.9200],
          [1.0400],
          [0.9600]]),
  'c_nmr': tensor([[153.4900],
          [135.5400],
          [134.6900],
          [125.9700],
          [120.9700],
          [108.6800],
          [ 53.9700],
          [ 44.4600],
          [ 43.3100],
          [ 29.5300],
          [ 25.6500],
          [ 22.4700],
          [ 21.6900],
          [ 19.2600],
          [ 18.6300]]),
  'hsqc': tensor([[ 1.0875e+02,  7.0500e+00,  3.8675e+03],
          [ 1.2100e+02,  6.8600e+00,  4.6854e+03],
          [ 5.3950e+01,  3.6600e+00, -5.7776e+03],
          [ 1.8820e+01,  2.7900e+00, -1.7028e+03],
          [ 4.3320e+01,  2.6800e+00,  3.1988e+03],
          [ 1.8820e+01,  2.6800e+00, -1.7261e+03],
          [ 2.2690e+01,  2.4700e+00,  6.1998e+03],
          [ 4.